In [11]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import h5py, cv2, joblib
from torchvision.models.feature_extraction import create_feature_extractor
from IPython.display import clear_output
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
from useful_stuff.general_utils.utils import TimeSeries, print_wise
from useful_stuff.image_processing.utils import load_torchvision_model, load_timm_model
from project_specific_utils.dataloader import load_eyetracking_data
from image_processing.utils import read_video
from image_processing.gaze_dep_models import sequential_gaze_dep_mod, save_ipca_patch

In [13]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    sub_num = 3
    fs = 24
    model_name = "alexnet";
    layer_name = "classifier.2";
    pkg = "torchvision"
    
    sq_side = 384
cfg = Cfg()

In [ ]:
def ANN_extraction_projection_1917_wrapper(paths: dict[str: str], rank: int, sub_num: int, model, sq_side: int, input_size, model_name: str, layer_name, n_components, pooling, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5,): 
    ipca_path = save_ipca_patch(paths, model_name, layer_name, n_components, sq_side, pooling,) # model_name, layer_name, n_components, sq_size, pooling
    ipca_obj = joblib.load(ipca_path)
    PCs = ipca_obj.components_.T
    print_wise(f"Start running {model_name} for sub {sub_num}", rank=rank)
    for irun in range(1, 7):
        gaze_dep_ANN_extraction(paths, rank, sub_num, sq_side, model, model_name, layer_name, n_components, pooling, PCs, input_size, irun, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5, )
    # end for irun in range(1, 7):
# EOF


# TODO run function to see if it's working (listen to audio)

In [47]:
from project_specific_utils.utils import run2part
from image_processing.gaze_dep_models import pad_frame, extract_square_patch, extract_features_1917_movie
from useful_stuff.image_processing.utils import get_video_dimensions, preprocess_batch
import torch
def gaze_dep_ANN_extraction(paths: dict[str: str], rank: int, sub_num: int, sq_side: int, model, model_name: str,layer_name, n_components, pooling, PCs, input_size,  run: int, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5,): 
    movie_part = run2part(run)
    movie_fn = f"{paths['data_path']}/stimuli/Project1917_movie_part{movie_part}_24Hz.mp4"
    cap = cv2.VideoCapture(movie_fn)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, round(5*fps))
    h, w, frames_n = get_video_dimensions(cap)
    save_name = save_ANN_features(paths, f"{model_name}_{layer_name}", round(fps), sub_num, run, n_components, sq_side, pooling,)
    if os.path.exists(save_name):
        print_wise(f"model already exists at {save_name}", rank=rank)
        return None
    # end if os.path.exists(save_name):
    feature_extractor = create_feature_extractor(
        model, return_nodes=[layer_name]
    ).to(device)
    xy_gaze, _ = load_eyetracking_data(paths, sub_num, run, eye_fs, xy=True)
    xy_gaze.resample(fps)
    frames_n -= round(secs_to_skip*fps) +2 # to be on the safe side, because when downsampled, the number of gaze-datapoints exceeds the number of frames
    if frames_n > len(xy_gaze):
        raise IndexError(f"The number of frames ({frames_n}) is larger than the number of gaze datapoints ({len(xy_gaze)}) in sub {sub_num} run {run}")
    # end if frames_n > len(xy_gaze):

    offset_dims = ((screen_res[0] -h)//2 , ( screen_res[1] - w)//2)
    canvas = None
    features = []
    for frame_idx in range(frames_n):
        xy = xy_gaze[frame_idx]
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError(f"Failed to read frame {frame_idx} from {movie_fn}")
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        canvas = pad_frame(frame, (h,w), offset_dims,)
        frame_patch = extract_square_patch(canvas, round(xy[0]), round(xy[1]), sq_side)
        frame_patch = torch.from_numpy(frame_patch)
        curr_features = extract_features_1917_movie(frame_patch[None, :, :, :], feature_extractor, layer_name, input_size, pooling=pooling, device=device)
        curr_features = np.squeeze(curr_features @ PCs) # TODO check if features are a column vector
        features.append(curr_features)
        if frame_idx%1000 == 0:
            print_wise(f"processed frame {frame_idx} of {frames_n} in {layer_name}")
        # canvas = sequential_gaze_dep_loop(cap, xy_gaze, frame_idx, sq_side, (h,w), offset_dims, canvas, features, func, rank, sub_num, run, *args, **kwargs)
    # end for frame_idx in range(frames_n):
    
    features = np.stack(features, axis=1)
    with h5py.File(save_name, "w") as f:
        f.create_dataset("vecrep", data=features)
    # end with h5py.File(save_name, "w") as f:
    print_wise(f"model {model_name} saved at {save_name}", rank=rank)
# EOF



In [48]:
if cfg.pkg =="torchvision":
    model = load_torchvision_model(cfg.model_name, device='mps')
elif cfg.pkg == "timm":
    model = load_timm_model(cfg.model_name, device='mps')

In [53]:
rank = 0; sub_num = 3;  n_components = 1000; pooling = "all"; PCs = "30"; input_size = 224; run = 1; eye_fs =50; device = "mps"

# gaze_dep_ANN_extraction(paths, rank, sub_num, cfg.sq_side, model, cfg.model_name, cfg.layer_name, n_components, pooling, PCs, input_size,  run, eye_fs, device, screen_res=(1080, 1920), secs_to_skip=5,)
ANN_extraction_projection_1917_wrapper(paths, rank, sub_num, model, cfg.sq_side, input_size, cfg.model_name, cfg.layer_name, n_components, pooling, eye_fs, device,)

10:14:19 - rank 0 Start running alexnet for sub 3
10:14:20 - processed frame 0 of 20950 in classifier.2
10:14:31 - processed frame 1000 of 20950 in classifier.2
10:14:41 - processed frame 2000 of 20950 in classifier.2
10:14:50 - processed frame 3000 of 20950 in classifier.2
10:14:58 - processed frame 4000 of 20950 in classifier.2
10:15:07 - processed frame 5000 of 20950 in classifier.2
10:15:15 - processed frame 6000 of 20950 in classifier.2
10:15:24 - processed frame 7000 of 20950 in classifier.2
10:15:32 - processed frame 8000 of 20950 in classifier.2
10:15:41 - processed frame 9000 of 20950 in classifier.2
10:15:52 - processed frame 10000 of 20950 in classifier.2
10:16:03 - processed frame 11000 of 20950 in classifier.2
10:16:14 - processed frame 12000 of 20950 in classifier.2
10:16:25 - processed frame 13000 of 20950 in classifier.2
10:16:35 - processed frame 14000 of 20950 in classifier.2
10:16:46 - processed frame 15000 of 20950 in classifier.2
10:16:57 - processed frame 16000 of